# BitCrusher URC ISE Demonstration

In [ ]:
import numpy as np
import chipwhisperer as cw
from time import sleep

from .mover import EnderMover
from device import Device  # BitCrusher

fault_locations: dict[str, tuple[float, float]] = {
    "reset": (44, 99.5),
    "halt": (42, 95.5)  # Close I think..?
}

def connect_husky():
    scope = cw.scope()
    scope.clock.adc_mul = 1
    scope.clock.clkgen_freq = 25E6

    scope.glitch.enabled = True

    scope.glitch.clk_src = "pll"
    scope.glitch.trigger_src = "ext_single"

    scope.clock.clkgen_freq = 25E6

    scope.io.tio4 = 'high_z'  # set trigger pin as input
    scope.trigger.module = 'basic'  # use basic (edge) triggering
    scope.trigger.triggers = 'tio4'  # set trigger module input to tio4
    scope.io.glitch_trig_mcx = 'trigger'  # output tirgger signal on glitch / trig SMB connector

    return scope

## Set up peripheral devices

In [ ]:
mover = EnderMover(port="/dev/ttyUSB0")
husky = connect_husky()

x_orig = 37.0
y_orig = 104.0
mover.g.move(x=x_orig, y=y_orig)
mover.g.sleep(duration=0)
mover.calibrate_z_min()

## Set up BitCrusher

In [ ]:
bitcrusher = Device()

## Fault 1: Reset

In [ ]:
# move to fault location
mover.g.move(x=fault_locations["reset"][0], y=fault_locations["reset"][1])
mover.g.sleep(duration=0)

# configure & arm bitcrusher
bitcrusher._write_arming_param("voltage", 300)
bitcrusher.arm()
sleep(2)

# Inject fault
for _ in range(10):
    sleep(0.01)
    # Activate AD3 pulse generation
    husky.io.tio1 = True
    sleep(0.001)
    husky.io.tio1 = False
    sleep(0.05)

bitcrusher.disarm()

## Fault 2: Halt

In [ ]:
mover.g.move(x=fault_locations["halt"][0], y=fault_locations["halt"][1])
mover.g.sleep(duration=0)

# configure & arm bitcrusher
bitcrusher._write_arming_param("voltage", 400)
bitcrusher.arm()
sleep(2)

# Inject fault
for _ in range(10):
    sleep(0.01)
    # Activate AD3 pulse generation
    husky.io.tio1 = True
    sleep(0.001)
    husky.io.tio1 = False
    sleep(0.05)

bitcrusher.disarm()